# Catching formula-level sabotage symbolically

[Auditing Sabotage Bench](https://arxiv.org/abs/2604.16286) (Redwood Research)
shows that LLM auditors struggle to detect sabotage in research code: the best
auditor reaches AUROC 0.77, because good sabotage *looks* plausible, and a
reader judges plausibility.

Several of the benchmark's sabotages are **mathematical substitutions** — an
entropy formula replaced by a linear approximation, an order-statistic index
clamped. This notebook shows how a symbolic lift turns detecting that class
from judgment into arithmetic.

The saboteur below "implements precision" but counts **false negatives**
instead of false positives. The code reads plausibly, and on balanced errors
it returns *the identical value* to the real metric — no spot check
distinguishes them.

In [1]:
import numpy as np
from sklearn.metrics import precision_score

def sabotaged_precision(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 0) & (y_true == 1))   # <- counts FN, not FP
    return tp / (tp + fp)

y_true = np.array([1, 1, 0, 0, 1, 0], dtype=float)
y_pred = np.array([1, 0, 1, 0, 1, 0], dtype=float)

sabotaged_precision(y_true, y_pred), precision_score(y_true, y_pred)

(np.float64(0.6666666666666666), 0.6666666666666666)

Identical outputs. A test suite built on cases like this passes.

Now lift the sabotaged function and read what it **actually computes**:

In [2]:
import skverify

out = skverify.to_sympy(sabotaged_precision, y_true, y_pred)
out.pretty()

't0      = Eq(y_true[j], 1)\nt1      = Sum(Piecewise((1, t0 & Eq(y_pred[j], 1)), (0, True)), (j, 0, 5))\n\nformula = t1/(t1 + Sum(Piecewise((1, t0 & Eq(y_pred[j], 0)), (0, True)), (j, 0, 5)))'

The certificate's denominator is
$\sum_j [\,y_{pred}[j] = 0 \wedge y_{true}[j] = 1\,]$ — **false negatives**.
The lifted formula is
$\mathrm{TP} / (\mathrm{TP} + \mathrm{FN})$: this function provably computes
**recall**, not precision, on *every* input — established from one trace,
with zero test cases.

For comparison, the honest metric lifts to the textbook formula:

In [3]:
def honest_precision(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    return tp / (tp + fp)

skverify.to_sympy(honest_precision, y_true, y_pred).pretty()

't0      = Eq(y_pred[j], 1)\nt1      = Sum(Piecewise((1, t0 & Eq(y_true[j], 1)), (0, True)), (j, 0, 5))\n\nformula = t1/(t1 + Sum(Piecewise((1, t0 & Eq(y_true[j], 0)), (0, True)), (j, 0, 5)))'

The two certificates look symmetric — the sabotage hides in the swapped
subscripts (`t0` conditions on `y_true` above, `y_pred` below). But an
auditor never needs to eyeball that: the certificates establish the
formulas differ, and a two-line search produces an input where the
behaviors visibly split:

In [4]:
import itertools

# the certificates established the FORMULAS differ; a witness shows
# the BEHAVIOR differs -- searched on the concrete lane, where
# division-by-zero corners evaluate the way numpy actually evaluates
import warnings
warnings.filterwarnings("ignore")  # nan corners in the search
for yt in itertools.product([0.0, 1.0], repeat=4):
    for yp in itertools.product([0.0, 1.0], repeat=4):
        a = sabotaged_precision(np.array(yt), np.array(yp))
        b = honest_precision(np.array(yt), np.array(yp))
        if np.isfinite(a) and np.isfinite(b) and a != b:
            print("VERDICT: behaviors differ. Witness:")
            print(" y_true    =", yt)
            print(" y_pred    =", yp)
            print(f" sabotaged = {a}   honest = {b}")
            break
    else:
        continue
    break

VERDICT: behaviors differ. Witness:
 y_true    = (0.0, 0.0, 0.0, 1.0)
 y_pred    = (0.0, 0.0, 1.0, 1.0)
 sabotaged = 1.0   honest = 0.5


An auditor comparing these two certificates against the stated methodology
("we report precision") doesn't weigh plausibility — one expression matches
the definition and one provably does not. The same lift works on unmodified
scikit-learn and statsmodels internals; see the
[coverage page](../doc/coverage.md) for what traces today.